# 손글씨 숫자 인식 (HNU Digits) — 최종 제출 노트북

| 항목 | 내용 |
|---|---|
| 팀명 | (리더보드 표시 이름) |
| 팀원 | 학번 이름, 학번 이름 |
| 팀원별 기여 | 예) 홍길동: EDA·전처리 / 김데이터: 모델 튜닝 |
| 리더보드 제출 번호 (있으면) | #( ) - 제출 이슈의 채점 댓글에 나온 "제출 번호" |
| 최종 Public 점수 | |
| 사용한 자료 (외부 데이터, 사전학습 모델, `test.csv` 라벨 등) | 없음 / (있으면 출처와 방법) |

**제출 규칙**: 최종(Private) 순위는 마감 후 이 노트북을 교수자가 **공개하지 않은 새 테스트 파일**로 다시 실행해 얻은 정확도로 정합니다.

- 테스트 데이터는 반드시 아래 `TEST_PATH`로만 읽으세요. 이 변수만 바꾸면 다른 테스트 파일에 대한 `submission.csv`가 만들어져야 합니다.
- Colab에서 "모두 실행" 한 번으로 오류 없이 끝나야 하고, 30분을 넘지 않아야 합니다. 학습을 포함해도 되고, 미리 학습해 둔 모델을 불러와도 됩니다.
- 공개된 자료(`train_hnu.csv`, `test.csv`, 공개 데이터셋, 사전학습 모델)는 어떤 방식으로든 써도 됩니다. `test.csv`에 라벨을 붙여 썼다면 무엇을 어떻게 했는지 위 표에 적으세요.
- 노트북이 쓰는 파일(직접 만든 라벨, 미리 학습한 모델 등)은 노트북과 같은 폴더에 두면 실행되도록 파일 이름만으로 읽고, 노트북과 함께 LMS에 제출하세요.
- 실행 중에 외부 AI 서비스를 호출하거나 테스트 데이터를 외부로 보내면 안 됩니다. 예측은 노트북 안의 코드와 모델이 만들어야 합니다.
- 필요 없는 셀은 지우고 섹션 제목은 유지해 주세요.

In [ ]:
# 0. 환경 설정 (수정하지 마세요)
import random, os, sys, platform
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sklearn

SEED = 42
random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.max_columns", 50)
print("python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)


## 1. 데이터 불러오기
`pixel0`~`pixel783`은 28×28 이미지를 행 우선으로 펼친 값(0~255)입니다.

In [ ]:
DATA = "https://gisup89.github.io/competition/competitions/hnu-digits/data/"
TRAIN_PATH = DATA + "train_hnu.csv"
TEST_PATH = DATA + "test.csv"   # 최종 평가 때 교수자가 이 경로만 다른 테스트 파일로 바꿔 실행합니다. 테스트 데이터는 이 변수로만 읽으세요.
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
X = train.filter(like="pixel").to_numpy(dtype=np.float32) / 255.0
y = train["label"].to_numpy()
X_test = test.filter(like="pixel").to_numpy(dtype=np.float32) / 255.0
print(X.shape, X_test.shape)
print("숫자별 개수:", dict(zip(*np.unique(y, return_counts=True))))

## 2. 데이터 살펴보기

In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(12, 4))
for d in range(10):
    idx = np.where(y == d)[0][:3]
    for r in range(3):
        ax = axes[r, d]; ax.axis("off")
        if r < len(idx):
            ax.imshow(X[idx[r]].reshape(28, 28), cmap="binary")
plt.suptitle("Samples per digit"); plt.show()

## 3. (선택) MNIST 불러오기
외부 데이터를 쓰려면 이 셀을 사용하고, 노트북 첫 셀의 표에 기록하세요. 우리 데이터와 획 굵기·기울기가 다르므로 섞어 쓰거나 전처리로 맞추는 실험이 필요합니다.

In [ ]:
# from sklearn.datasets import fetch_openml
# mnist = fetch_openml("mnist_784", as_frame=False)
# X_mnist = mnist.data.astype(np.float32) / 255.0
# y_mnist = mnist.target.astype(int)
# print(X_mnist.shape)

## 4. 검증 전략
층화 교차검증으로 모델을 비교합니다. 테스트 데이터는 학습 데이터에 없는 사람의 글씨이므로, 교차검증 점수가 리더보드 점수보다 높게 나오는 것이 보통입니다.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def acc_cv(model, X=X, y=y):
    scores = cross_val_score(model, X, y, scoring="accuracy", cv=cv)
    print(f"CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}")
    return scores

## 5. 모델 비교

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

models = {
    "LogisticRegression": LogisticRegression(max_iter=2000),
    "KNN(k=3)": KNeighborsClassifier(3),
    "RandomForest": RandomForestClassifier(300, random_state=SEED, n_jobs=-1),
    "SVC(rbf)": SVC(C=5, gamma="scale"),
}
for name, m in models.items():
    print(name, end=" | ")
    acc_cv(m)

## 6. 오차 분석
오차 행렬로 어떤 숫자끼리 헷갈리는지 보고, 틀린 이미지를 직접 확인합니다.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

model = SVC(C=5, gamma="scale")
y_pred = cross_val_predict(model, X, y, cv=cv)
ConfusionMatrixDisplay.from_predictions(y, y_pred); plt.show()
print(classification_report(y, y_pred, digits=3))

## 7. 튜닝 · 데이터 증식
`GridSearchCV`로 하이퍼파라미터를 고릅니다. 이미지를 1~2픽셀 이동시키는 증식 예시는 주석으로 넣어 두었으니, 효과가 있는지 직접 검증해 보고 쓰세요.

In [ ]:
from sklearn.model_selection import GridSearchCV
from scipy.ndimage import shift

# 학습 데이터가 6,820장이므로 3겹 교차검증으로 시간을 줄입니다
search = GridSearchCV(SVC(), {"C": [3, 10], "gamma": ["scale"]}, scoring="accuracy", cv=3, n_jobs=-1)
search.fit(X, y)
print("best:", search.best_params_, f"{search.best_score_:.4f}")
final_model = search.best_estimator_
X_final, y_final = X, y

# (선택) 데이터 증식 예시: 상하좌우 1픽셀 이동으로 학습 데이터를 5배로 늘립니다 (학습 시간도 늘어납니다)
# def shift_image(img784, dx, dy):
#     return shift(img784.reshape(28, 28), [dy, dx], cval=0).reshape(784)
# X_aug = [X]; y_aug = [y]
# for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
#     X_aug.append(np.array([shift_image(im, dx, dy) for im in X])); y_aug.append(y)
# X_final, y_final = np.concatenate(X_aug), np.concatenate(y_aug)
# print(X_final.shape)

## 8. 최종 학습 · 제출 파일 생성

In [ ]:
final_model.fit(X_final, y_final)
pred = final_model.predict(X_test).astype(int)
submission = pd.DataFrame({"id": test["id"], "label": pred})
assert len(submission) == len(test) and submission["label"].between(0, 9).all()
submission.to_csv("submission.csv", index=False)
print(submission["label"].value_counts().sort_index())
print("submission.csv 저장 완료 — 사이트의 '제출 파일 암호화' 페이지에서 변환해 제출하세요.")

## 9. 정리
- **접근 방법 요약**:
- **효과가 있었던 것 / 없었던 것**:
- **검증 점수와 리더보드 점수 비교**:
- **더 해 보고 싶은 것**: